In [1]:
import os

In [2]:
%pwd

'c:\\Users\\aabhi\\Desktop\\kidney_disease\\Kidney-Disease-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\aabhi\\Desktop\\kidney_disease\\Kidney-Disease-Classification'

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_dir : Path
    updated_base_model_path : Path
    params_image_size : list
    params_learning_rate: float
    params_include_top: bool
    params_weights:str
    params_classes:int

In [13]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(
            self,
            config_file_path = CONFIG_FILE_PATH,
            params_file_path = PARAMS_FILE_PATH):

            self.config = read_yaml(config_file_path)
            self.params = read_yaml(params_file_path)

            create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self)-> PrepareBaseModelConfig:
          config = self.config.prepare_base_model

          create_directories([config.root_dir])
          prepare_base_model_config = PrepareBaseModelConfig(
              root_dir = config.root_dir,
              base_model_dir = config.base_model_dir,
              updated_base_model_path = config.updated_base_model_path,
              params_image_size = self.params.IMAGE_SIZE,
              params_learning_rate = self.params.LEARNING_RATE,
              params_include_top = self.params.INCLUDE_TOP,
              params_weights = self.params.WEIGHTS,
              params_classes = self.params.CLASSES                
          )

          return prepare_base_model_config


    

In [15]:
import os 
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [ ]:
import tensorflow as tf


class PrepareBaseModel:

    def __init__(self, config):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.MobileNetV2(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top,
        )

        self.save_model(
            path=self.config.base_model_dir,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(
        model,
        classes,
        freeze_all,
        freeze_till,
        learning_rate
    ):

        # Freeze all layers
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False

        # Freeze layers till a certain index
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:freeze_till]:
                layer.trainable = False

        # Add custom classification head
        flatten_in = tf.keras.layers.Flatten()(model.output)

        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        # Create full model
        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        # Compile model
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()

        return full_model




    @staticmethod
    def save_model(path:Path,model:tf.keras.Model):
        model.save(path)

